# Appendix Tables — WaveletFlow Paper

## 0. Imports & Global Settings

In [138]:
import glob
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

rc = {
    "axes.axisbelow":      True,
    "axes.linewidth":      1.0,
    "axes.labelsize":      "x-small",
    "axes.titlesize":      "x-small",
    "figure.dpi":          150,
    "figure.figsize":      (3.2, 3.2),
    "font.size":           12.0,
    "legend.fontsize":     "xx-small",
    "lines.linewidth":     1.0,
    "savefig.transparent": True,
    "xtick.labelsize":     "xx-small",
    "xtick.major.width":   1.0,
    "ytick.labelsize":     "xx-small",
    "ytick.major.width":   1.0,
    "font.family":         ["serif"],
    "text.usetex":         True,
    "text.latex.preamble": r"\usepackage{amsfonts}",
}
plt.rcParams.update(rc)

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## 1. Model Registry & Derived Sets

In [139]:
def rgb(r, g, b): return f"#{r:02x}{g:02x}{b:02x}"

# Each entry: label, color, dash, lw, group in {wfm-1scale, wfm-3scale, generative, deterministic}
MODEL_REGISTRY = {
    # -- WFM 1-scale variants --------------------------------------------------
    "WaveletFlow_1scale_haar_v4.0": dict(label="WFM-1 haar", color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db2_v4.0":  dict(label="WFM-1 db2",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db4_v4.0":  dict(label="WFM-1 db4",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db6_v4.0":  dict(label="WFM-1 db6",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    # -- WFM 3-scale variants --------------------------------------------------
    "WaveletFlow_3scales_haar_v4.0": dict(label="WFM-3 haar", color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db2_v4.0":  dict(label="WFM-3 db2",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db4_v4.0":  dict(label="WFM-3 db4",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db6_v4.0":  dict(label="WFM-3 db6",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    # -- Generative baselines --------------------------------------------------
    "FM_data_space_v4.0":       dict(label=r"FM$_\text{pixel}$",   color=rgb(176,176,176), dash=(1,0), lw=1.8, group="generative"),
    "FourierFlow_32modes_v4.0": dict(label=r"FM$_\text{fourier}$", color=rgb(143, 94,106), dash=(1,0), lw=1.8, group="generative"),
    "WDNO":                     dict(label="WDNO",                  color=rgb(191,134,148), dash=(1,0), lw=1.8, group="generative"),
    # -- Deterministic baselines -----------------------------------------------
    "WNO_3scales_db2": dict(label="WNO",  color=rgb(176,136, 84), dash=(1,0), lw=1.8, group="deterministic"),
    "FNO":             dict(label="FNO",  color=rgb(208,169,110), dash=(1,0), lw=1.8, group="deterministic"),
    "TFNO":            dict(label="TFNO", color=rgb(236,227,163), dash=(1,0), lw=1.8, group="deterministic"),
}

all_models = list(MODEL_REGISTRY)
WFM_1scale = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "wfm-1scale"]
WFM_3scale = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "wfm-3scale"]
WFM_all    = WFM_1scale + WFM_3scale
gen_models = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "generative"]
det_models = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "deterministic"]

# Canonical table ordering: WFM-1scale -> WFM-3scale -> generative -> deterministic
MODEL_ORDER = WFM_1scale + WFM_3scale + gen_models + det_models

labels     = {m: MODEL_REGISTRY[m]["label"] for m in all_models}
palette    = {m: MODEL_REGISTRY[m]["color"] for m in all_models}
dashes     = {m: MODEL_REGISTRY[m]["dash"]  for m in all_models}
linewidths = {m: MODEL_REGISTRY[m]["lw"]    for m in all_models}

GT_LABEL = "Ground-truth"
palette[GT_LABEL] = "#222222"; dashes[GT_LABEL] = (1, 0)
labels[GT_LABEL]  = GT_LABEL;  linewidths[GT_LABEL] = 1.8

def model_group(m):
    if m in WFM_1scale: return "wfm-1scale"
    if m in WFM_3scale: return "wfm-3scale"
    if m in gen_models: return "generative"
    return "deterministic"

## 2. PDE Metadata

In [140]:
PDEs = ["turbulent_radiative_layer_2D", "rayleigh_benard", "active_matter"]

PDE_LABELS = {
    "turbulent_radiative_layer_2D": "Turbulence",
    "rayleigh_benard":              r"Rayleigh--B\'{e}nard",
    "active_matter":                "Active Matter",
}

# Physical parameters varied in ablation studies
PDE_CONSTANTS = {
    "turbulent_radiative_layer_2D": ["tcool"],
    "rayleigh_benard":              ["Rayleigh", "Prandtl"],
    "active_matter":                ["zeta", "alpha"],
}

# Raw horizon strings produced by the step->horizon mapping (per PDE)
HORIZON_BINS_RAW = {
    "turbulent_radiative_layer_2D": ["01:32", "33:64", "65:97"],
    "rayleigh_benard":              ["01:32", "33:64", "65:196"],
    "active_matter":                ["01:16", "17:32", "33:77"],
}

# Generic column labels for unified multi-PDE tables (Short / Mid / Long)
GENERIC_HORIZONS = ["Short", "Mid", "Long"]
HORIZON_TO_GENERIC = {
    pde: dict(zip(raw, GENERIC_HORIZONS))
    for pde, raw in HORIZON_BINS_RAW.items()
}

# Spectral frequency bands
BANDS     = ["Low", "Mid", "High"]
BAND_COLS = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}

# Horizon caption note (reused in LaTeX exports)
HZ_CAPTION_NOTE = (
    r"Short/Mid/Long $\equiv$ [1\!:\!32]/[33\!:\!64]/[65\!:\!97] (Turbulence), "
    r"[1\!:\!32]/[33\!:\!64]/[65\!:\!196] (Rayleigh--B\'enard), "
    r"[1\!:\!16]/[17\!:\!32]/[33\!:\!77] (Active Matter)."
)

## 3. Data Loading

In [141]:
LOAD_STATS = True
LOAD_PSD   = False


def enrich(df, stem, model_name, dataset_name):
    # Preserve GT rows already labelled in the CSV
    parts     = stem.split("_")
    nsampling = next((p.replace("nsampling", "") for p in parts if p.startswith("nsampling")), None)
    nens      = next((p.replace("nens",      "") for p in parts if p.startswith("nens")),      None)
    if "model" in df.columns:
        df.loc[df["model"] != GT_LABEL, "model"] = model_name
    else:
        df["model"] = model_name
    df["source"]    = stem
    df["dataset"]   = dataset_name
    df["pde"]       = dataset_name
    df["nsampling"] = int(nsampling) if nsampling else None
    df["nens"]      = int(nens)      if nens      else None
    return df


DEDUP_EXCLUDE = {"source", "nsampling", "nens"}

dfs_stats, dfs_psd = [], []
for pde in PDEs:
    print(f"\n{'='*60}\nLoading: {pde}\n{'='*60}")
    for model in all_models:
        for suffix, target, flag in [
            ("stats", dfs_stats, LOAD_STATS),
            ("psd",   dfs_psd,   LOAD_PSD),
        ]:
            if not flag:
                continue
            pattern = f"{PROJECT_ROOT}/experiments/{pde}/metrics/{model}*{suffix}.csv"
            matched = sorted(glob.glob(pattern))
            if not matched:
                print(f"  [MISS {suffix}] {model}")
            for fpath in matched:
                try:
                    df = pd.read_csv(fpath)
                    target.append(enrich(df, Path(fpath).stem, model, pde))
                    print(f"  [OK   {suffix}] {Path(fpath).name}")
                except Exception as exc:
                    print(f"  [ERR] {fpath}: {exc}")


def _concat_dedup(dfs):
    if not dfs:
        return pd.DataFrame()
    full = pd.concat(dfs, ignore_index=True)
    return full.drop_duplicates(
        subset=[c for c in full.columns if c not in DEDUP_EXCLUDE]
    )


stats_df = _concat_dedup(dfs_stats)
psd_df   = _concat_dedup(dfs_psd)
print(f"\nstats_df : {stats_df.shape}   cols: {list(stats_df.columns)}")
print(f"psd_df   : {psd_df.shape}   cols: {list(psd_df.columns)}")


Loading: turbulent_radiative_layer_2D
  [OK   stats] WaveletFlow_1scale_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] FM_data_space_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] FourierFlow_32modes_v4.0_nsampling50_nens8_turbul

## 4. Feature Engineering

In [142]:
AGG_MODE = "mean"   # "mean" | "median"

def aggregate(x):
    return np.nanmean(x) if AGG_MODE == "mean" else np.nanmedian(x)


def _horizon_fn(bins):
    # Build step -> horizon_label function from ordered (threshold, label) pairs
    def fn(t):
        for thresh, label in bins:
            if t <= thresh:
                return label
        return bins[-1][1]
    return fn


HORIZON_FN = {
    "turbulent_radiative_layer_2D": _horizon_fn([(32,"01:32"), (64,"33:64"), (97, "65:97")]),
    "rayleigh_benard":              _horizon_fn([(32,"01:32"), (64,"33:64"), (196,"65:196")]),
    "active_matter":                _horizon_fn([(16,"01:16"), (32,"17:32"), (77, "33:77")]),
}

if LOAD_STATS and not stats_df.empty:
    stats_df = stats_df.copy()
    stats_df["step"] += 1
    stats_df["horizon"] = pd.NA
    for pde, fn in HORIZON_FN.items():
        mask = stats_df["pde"] == pde
        stats_df.loc[mask, "horizon"] = stats_df.loc[mask, "step"].apply(fn)
    stats_df["model_group"] = stats_df["model"].apply(model_group)
    if {"spread", "rmse"}.issubset(stats_df.columns):
        stats_df["spread_skill"] = (stats_df["spread"] + 1e-3) / (stats_df["rmse"] + 1e-3)

if LOAD_PSD and not psd_df.empty:
    psd_df = psd_df.copy()
    psd_df["step"] += 1
    psd_df["horizon"] = pd.NA
    for pde, fn in HORIZON_FN.items():
        mask = psd_df["pde"] == pde
        psd_df.loc[mask, "horizon"] = psd_df.loc[mask, "step"].apply(fn)
    psd_df["model_group"] = psd_df["model"].apply(model_group)

## 5. Styling & LaTeX Helpers

In [143]:
# ============================================================
# Jupyter display helpers
# ============================================================

def highlight_min(df: pd.DataFrame) -> pd.DataFrame:
    # Bold minimum, underline 2nd minimum, per column (globally across all rows)
    out = pd.DataFrame("", index=df.index, columns=df.columns)
    for col in df.columns:
        vals = df[col].dropna().sort_values()
        if len(vals) >= 1: out.loc[vals.index[0], col] = "font-weight: bold"
        if len(vals) >= 2: out.loc[vals.index[1], col] = "text-decoration: underline"
    return out


def highlight_min_pde(df: pd.DataFrame) -> pd.DataFrame:
    # Bold minimum, underline 2nd minimum, per column, WITHIN each PDE group
    # (for MultiIndex row DataFrames only)
    out = pd.DataFrame("", index=df.index, columns=df.columns)
    for pde in df.index.get_level_values(0).unique():
        mask = df.index.get_level_values(0) == pde
        sub  = df.loc[mask]
        for col in df.columns:
            vals = sub[col].dropna().sort_values()
            if len(vals) >= 1: out.loc[vals.index[0], col] = "font-weight: bold"
            if len(vals) >= 2: out.loc[vals.index[1], col] = "text-decoration: underline"
    return out


# ============================================================
# LaTeX helpers
# ============================================================

def _fmt_cell(val: float, rank: int) -> str:
    s = f"{val:.3f}"
    if rank == 1: return r"\textbf{" + s + "}"
    if rank == 2: return r"\underline{" + s + "}"
    return s


def to_latex_wide(
    df: pd.DataFrame,
    caption: str,
    label: str,
    model_groups: list = None,
    row_label: str = "Rollout",
    minimize: bool = True,
    placement: str = "ht",
) -> str:
    # Export a wide-format multi-PDE table to LaTeX (booktabs).
    #
    # Expected DataFrame layout
    # --------------------------
    # Index  : flat model display labels (strings), ordered as desired.
    # Columns: pd.MultiIndex with levels (pde_key, horizon_str).
    #          e.g. ("turbulent_radiative_layer_2D", "01:32")
    #
    # Visual structure produced
    # -------------------------
    #   \caption / \label
    #   \begin{tabular}{lccc ccc ccc}
    #   \toprule
    #     & \multicolumn{3}{c}{Turb. Rad. Layer} & ...
    #   \cmidrule(lr){2-4} \cmidrule(lr){5-7} ...
    #     Rollout & 01:32 & 33:64 & 65:97 & ...
    #   \midrule
    #     <model group 1 rows>
    #   \midrule
    #     <model group 2 rows>
    #   \bottomrule
    #
    # Parameters
    # ----------
    # model_groups : list of lists of model label strings, defining \midrule
    #                separators between row groups.  Defaults to a single group
    #                (no separators).  Models absent from df are silently skipped.
    # row_label    : text for the leftmost header cell (default "Rollout").
    # minimize     : if True, bold = smallest value per column; else largest.
    # placement    : LaTeX float placement string (default "ht").

    assert isinstance(df.columns, pd.MultiIndex) and df.columns.nlevels == 2, (
        "df.columns must be a 2-level MultiIndex (pde_key, horizon_str). "
        "Use build_main_table(raw_horizons=True) to obtain this layout."
    )

    pdes     = list(dict.fromkeys(df.columns.get_level_values(0)))  # preserve order
    horizons = {pde: [h for p, h in df.columns if p == pde] for pde in pdes}
    n_per_pde = {pde: len(horizons[pde]) for pde in pdes}
    n_cols   = sum(n_per_pde.values())

    # -- Column ranks (global: across ALL model rows, per column) --------------
    ranks = {}
    for col in df.columns:
        sorted_idx = df[col].dropna().sort_values(ascending=minimize).index.tolist()
        ranks[col] = {m: i + 1 for i, m in enumerate(sorted_idx)}

    # -- Column spec -----------------------------------------------------------
    col_spec = "l" + "".join(" " + "c" * n for n in n_per_pde.values())

    # -- PDE header row --------------------------------------------------------
    start = 2   # 1-indexed; column 1 is the model label
    cmidrules = []
    pde_cells = [row_label]
    for pde in pdes:
        n = n_per_pde[pde]
        pde_cells.append(r"\multicolumn{" + str(n) + r"}{c}{" + PDE_LABELS.get(pde, pde) + "}")
        cmidrules.append(r"\cmidrule(lr){" + str(start) + "-" + str(start + n - 1) + "}")
        start += n
    header_pde  = " & ".join(pde_cells) + r" \\"
    header_cmir = " ".join(cmidrules)

    # -- Horizon sub-header row ------------------------------------------------
    hz_cells = [""]
    for pde in pdes:
        hz_cells.extend(horizons[pde])
    header_hz = " & ".join(hz_cells) + r" \\"

    # -- Model row groups ------------------------------------------------------
    if model_groups is None:
        model_groups = [list(df.index)]

    data_lines = []
    for g_idx, group in enumerate(model_groups):
        if g_idx > 0:
            data_lines.append(r"\midrule")
        for model in group:
            if model not in df.index:
                continue
            row   = df.loc[model]
            cells = [model]
            for col in df.columns:
                val  = row[col]
                rank = ranks[col].get(model, 99)
                cells.append("--" if pd.isna(val) else _fmt_cell(val, rank))
            data_lines.append(" & ".join(cells) + r" \\")

    # -- Assemble --------------------------------------------------------------
    lines = [
        r"\begin{table}[" + placement + "]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        header_pde,
        header_cmir,
        header_hz,
        r"\midrule",
        *data_lines,
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)


def to_latex_ablation(
    df: pd.DataFrame,
    caption: str,
    label: str,
    param_label: str = None,
    minimize: bool = True,
    placement: str = "ht",
) -> str:
    # Export a flat ablation table (model rows x param-value columns).
    # Bold = best, underline = 2nd best, per column (globally).
    #
    # Parameters
    # ----------
    # param_label : display name for the column header row (e.g. r"$Ra$").
    #               Defaults to df.columns.name if set, else empty string.
    if param_label is None:
        param_label = str(df.columns.name) if df.columns.name else ""

    ranks = {}
    for col in df.columns:
        sorted_idx = df[col].dropna().sort_values(ascending=minimize).index.tolist()
        ranks[col] = {m: i + 1 for i, m in enumerate(sorted_idx)}

    n_cols     = len(df.columns)
    col_spec   = "l" + "c" * n_cols
    header_par = param_label + " & " + " & ".join(str(c) for c in df.columns) + r" \\"

    lines = [
        r"\begin{table}[" + placement + "]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        header_par,
        r"\midrule",
    ]
    for model, row in df.iterrows():
        cells = [str(model)]
        for col in df.columns:
            val  = row[col]
            rank = ranks[col].get(model, 99)
            cells.append("--" if pd.isna(val) else _fmt_cell(val, rank))
        lines.append(" & ".join(cells) + r" \\")
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)

def to_latex_spectral_ablation(
    df: pd.DataFrame,
    caption: str,
    label: str,
    param_label: str = "",
    minimize: bool = True,
    placement: str = "ht",
) -> str:
    # Compact spectral ablation table.
    #
    # Input df  (output of build_spectral_ablation_table):
    #   index   : model display labels
    #   columns : (band, param_value) MultiIndex
    #
    # Output layout:
    #   rows    : (model, band) — model name spans 3 rows via \multirow,
    #             \midrule between models
    #   columns : param values
    #
    #   e.g.
    #   \toprule
    #   Model  & Band & 0.01 & 0.02 & 0.03 \\
    #   \midrule
    #   \multirow{3}{*}{WFM-1 haar} & Low  & bold{0.12} & 0.15 & 0.18 \\
    #                               & Mid  & ...
    #                               & High & ...
    #   \midrule
    #   ...

    assert isinstance(df.columns, pd.MultiIndex) and df.columns.nlevels == 2

    bands      = list(dict.fromkeys(df.columns.get_level_values(0)))  # preserve order
    param_vals = list(dict.fromkeys(df.columns.get_level_values(1)))  # preserve order

    # Per-(band, param_val) column ranks, global across all models
    ranks = {}
    for col in df.columns:
        sorted_idx = df[col].dropna().sort_values(ascending=minimize).index.tolist()
        ranks[col] = {m: i + 1 for i, m in enumerate(sorted_idx)}

    n_param  = len(param_vals)
    col_spec = "ll" + "c" * n_param
    header   = (
        r"Model & Band & "
        + " & ".join(str(v) for v in param_vals)
        + r" \\"
    )
    if param_label:
        header = param_label + " & & " + " & ".join(str(v) for v in param_vals) + r" \\"

    lines = [
        r"\begin{table}[" + placement + "]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        header,
        r"\midrule",
    ]

    models = list(df.index)
    for m_idx, model in enumerate(models):
        if m_idx > 0:
            lines.append(r"\midrule")
        model_cell = r"\multirow{" + str(len(bands)) + r"}{*}{" + model + "}"
        for b_idx, band in enumerate(bands):
            band_cell  = band
            model_part = model_cell if b_idx == 0 else ""
            cells      = [model_part, band_cell]
            for pv in param_vals:
                col  = (band, pv)
                val  = df.loc[model, col] if col in df.columns else float("nan")
                rank = ranks.get(col, {}).get(model, 99)
                cells.append("--" if pd.isna(val) else _fmt_cell(val, rank))
            lines.append(" & ".join(cells) + r" \\")

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)

## 6. Table Builders

In [144]:
# ============================================================
# Main appendix tables
# ============================================================

def build_main_table(
    metric: str,
    models: list = None,
    pdes: list = None,
    raw_horizons: bool = False,
) -> pd.DataFrame:
    # Build a unified multi-PDE appendix table.
    #
    # Parameters
    # ----------
    # metric        : stats_df column to aggregate (e.g. "vrmse", "crps").
    # models        : ordered model keys; defaults to MODEL_ORDER.
    # pdes          : ordered PDE keys;   defaults to PDEs.
    # raw_horizons  : if False (default), columns are Short/Mid/Long and the
    #                 index is a (PDE_label, model_label) MultiIndex, suitable
    #                 for Jupyter display.
    #                 if True, columns are a (pde_key, raw_horizon_str) MultiIndex
    #                 and the index is flat model display labels -- the layout
    #                 expected by to_latex_wide().
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon", metric]]
            .groupby(["model", "horizon"])[metric]
            .agg(aggregate).round(3).reset_index()
        )
        if raw_horizons:
            # Keep raw horizon strings; produce (pde_key, hz) MultiIndex columns
            sub = sub.pivot(index="model", columns="horizon", values=metric)
            raw_order = [h for h in HORIZON_BINS_RAW[pde] if h in sub.columns]
            sub = sub.reindex(
                index=[m for m in models if m in sub.index],
                columns=raw_order,
            )
            sub.index   = [labels.get(m, m) for m in sub.index]
            sub.columns = pd.MultiIndex.from_tuples(
                [(pde, h) for h in sub.columns], names=["pde", "horizon"]
            )
        else:
            # Map to generic Short/Mid/Long; produce (PDE_label, model) MultiIndex rows
            sub["horizon"] = sub["horizon"].map(HORIZON_TO_GENERIC[pde])
            sub = sub.pivot(index="model", columns="horizon", values=metric)
            sub = sub.reindex(
                index=[m for m in models if m in sub.index],
                columns=[h for h in GENERIC_HORIZONS if h in sub.columns],
            )
            sub.index = pd.MultiIndex.from_tuples(
                [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
                names=["PDE", "Model"],
            )
        frames.append(sub)

    result = pd.concat(frames, axis=1 if raw_horizons else 0)
    if raw_horizons:
        result.index.name = "Model"
    return result


def build_spectral_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Multi-PDE spectral coherence RMSE table (averaged over horizons).
    # Rows : (PDE, model) MultiIndex
    # Cols : Low / Mid / High
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon"] + list(BAND_COLS.keys())]
            .groupby(["model", "horizon"])[list(BAND_COLS.keys())]
            .agg(aggregate).reset_index()
            .rename(columns=BAND_COLS)
            .groupby("model")[BANDS]
            .mean().round(3)
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_spectral_horizon_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Per-PDE spectral table broken down by horizon AND band.
    # Rows : (PDE, model) MultiIndex
    # Cols : (band, horizon) MultiIndex
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon"] + list(BAND_COLS.keys())]
            .melt(id_vars=["model", "horizon"],
                  value_vars=list(BAND_COLS.keys()),
                  var_name="band_key", value_name="srmse")
            .assign(band=lambda x: x["band_key"].map(BAND_COLS),
                    horizon=lambda x: x["horizon"].map(HORIZON_TO_GENERIC[pde]))
            .groupby(["model", "band", "horizon"])["srmse"]
            .agg(aggregate).round(3).reset_index()
            .pivot(index="model", columns=["band", "horizon"], values="srmse")
            .reindex(columns=BANDS, level="band")
            .reindex(columns=GENERIC_HORIZONS, level="horizon")
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_w1_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Wasserstein-1 table: (PDE, model) rows x single W1 column.
    if models is None: models = WFM_all + gen_models
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "w1"]]
            .groupby("model")["w1"]
            .agg(aggregate).round(3)
            .reset_index().set_index("model")
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        sub.columns = [r"$W_1$"]
        frames.append(sub)
    return pd.concat(frames)


# ============================================================
# Ablation tables
# ============================================================

def build_ablation_table(
    metric: str,
    pde: str,
    param: str,
    models: list = None,
    param_filter=None,
) -> pd.DataFrame:
    # Ablation table: model rows x param-value columns.
    #
    # metric       : "vrmse", "crps", etc.
    # pde          : PDE key string.
    # param        : physical parameter column (e.g. "tcool", "Rayleigh", "alpha").
    # models       : models to show; defaults to WFM_all + gen_models.
    # param_filter : optional callable applied to param values,
    #                e.g. ``lambda v: v <= 0.03`` to restrict tcool range.
    if models is None: models = WFM_all + gen_models

    sub = (
        stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
        [["model", param, metric]]
        .dropna(subset=[param, metric])
    )
    if param_filter is not None:
        sub = sub[sub[param].apply(param_filter)]

    sub = (
        sub.groupby(["model", param])[metric]
        .agg(aggregate).round(3).reset_index()
        .pivot(index="model", columns=param, values=metric)
    )
    sub = sub.reindex([m for m in models if m in sub.index])
    sub.index = [labels.get(m, m) for m in sub.index]
    sub.index.name  = "Model"
    sub.columns.name = param
    return sub


def build_spectral_ablation_table(
    pde: str,
    param: str,
    models: list = None,
    param_filter=None,
) -> pd.DataFrame:
    # Spectral ablation table: model rows x (band, param_value) columns.
    if models is None: models = WFM_all + gen_models

    sub = (
        stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
        [["model", param] + list(BAND_COLS.keys())]
        .dropna(subset=[param])
    )
    if param_filter is not None:
        sub = sub[sub[param].apply(param_filter)]

    sub = (
        sub.melt(id_vars=["model", param],
                 value_vars=list(BAND_COLS.keys()),
                 var_name="band_key", value_name="srmse")
        .assign(band=lambda x: x["band_key"].map(BAND_COLS))
        .groupby(["model", "band", param])["srmse"]
        .agg(aggregate).round(3).reset_index()
        .pivot(index="model", columns=["band", param], values="srmse")
        .reindex(columns=BANDS, level="band")
    )
    sub = sub.reindex([m for m in models if m in sub.index])
    sub.index = [labels.get(m, m) for m in sub.index]
    sub.index.name = "Model"
    return sub

---
# A. Main Appendix Tables

**Row structure**: flat model labels, ordered within configurable `\midrule`-separated groups.
Group 1 = WFM (ours). Group 2 = other generative + deterministic baselines. Group 3 = FM_pixel.

**Column structure**: PDEs as `\multicolumn` headers; raw horizon strings underneath.

**Highlighting**: **bold** = best, underline = 2nd best, ranked *globally* per column.

In [145]:
# Row groups for the main appendix tables (each inner list = one \midrule-delimited block).
MODEL_GROUPS = [
    WFM_1scale + WFM_3scale,                                              # WFM (ours)
    [m for m in gen_models if m != "FM_data_space_v4.0"] + det_models,   # other baselines
    ["FM_data_space_v4.0"],                                               # FM_pixel last
]
MODEL_GROUPS_LABELS = [[labels[m] for m in g if m in labels] for g in MODEL_GROUPS]

## A.1  VRMSE -- All Models

In [146]:
t_vrmse = build_main_table(metric="vrmse", models=MODEL_ORDER, raw_horizons=True)
t_vrmse.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--")

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_69233/3832332138.py:4: RuntimeWarning: Mean of empty slice
  return np.nanmean(x) if AGG_MODE == "mean" else np.nanmedian(x)


In [147]:
print(to_latex_wide(
    t_vrmse,
    caption=r"VRMSE $\downarrow$ across baselines and PDEs. \textbf{Bold}: best; \underline{underline}: 2nd best per column.",
    label="tab:appendix_vrmse",
    model_groups=MODEL_GROUPS_LABELS,
    row_label="Rollout",
))

\begin{table}[ht]
\centering
\caption{VRMSE $\downarrow$ across baselines and PDEs. \textbf{Bold}: best; \underline{underline}: 2nd best per column.}
\label{tab:appendix_vrmse}
\begin{tabular}{l ccc ccc ccc}
\toprule
Rollout & \multicolumn{3}{c}{Turbulence} & \multicolumn{3}{c}{Rayleigh--B\'{e}nard} & \multicolumn{3}{c}{Active Matter} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
 & 01:32 & 33:64 & 65:97 & 01:32 & 33:64 & 65:196 & 01:16 & 17:32 & 33:77 \\
\midrule
WFM-1 haar & \underline{0.545} & 0.765 & 0.782 & 1.778 & 1.985 & 0.811 & 2.864 & 1.547 & 1.074 \\
WFM-1 db2 & 0.553 & 0.752 & 0.856 & 1.326 & 1.893 & 0.797 & 5.641 & 2.940 & 1.623 \\
WFM-1 db4 & 0.556 & \underline{0.731} & \textbf{0.776} & 1.732 & 2.362 & 0.800 & 14.219 & 5.562 & 2.490 \\
WFM-1 db6 & \textbf{0.544} & \textbf{0.705} & 0.787 & 1.463 & 1.998 & 0.980 & 6.782 & 2.932 & 1.567 \\
WFM-3 haar & 0.557 & 0.859 & 0.810 & 1.584 & 2.366 & 0.803 & \underline{0.849} & \underline{1.325} & \textbf{1.019} \\
WFM-

## A.2  CRPS -- WFM + Generative Models

In [148]:
t_crps = build_main_table(metric="crps", models=WFM_all + gen_models, raw_horizons=True)
t_crps.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--")

In [149]:
_crps_groups = [
    [labels[m] for m in WFM_1scale + WFM_3scale  if labels[m] in t_crps.index],
    [labels[m] for m in gen_models if m != "FM_data_space_v4.0" and labels[m] in t_crps.index],
    [labels["FM_data_space_v4.0"]],
]
print(to_latex_wide(
    t_crps,
    caption=r"CRPS $\downarrow$ across baselines and PDEs.",
    label="tab:appendix_crps",
    model_groups=_crps_groups,
    row_label="Rollout",
))

\begin{table}[ht]
\centering
\caption{CRPS $\downarrow$ across baselines and PDEs.}
\label{tab:appendix_crps}
\begin{tabular}{l ccc ccc ccc}
\toprule
Rollout & \multicolumn{3}{c}{Turbulence} & \multicolumn{3}{c}{Rayleigh--B\'{e}nard} & \multicolumn{3}{c}{Active Matter} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
 & 01:32 & 33:64 & 65:97 & 01:32 & 33:64 & 65:196 & 01:16 & 17:32 & 33:77 \\
\midrule
WFM-1 haar & 0.503 & 1.049 & 1.193 & 0.005 & 0.036 & 0.078 & 0.118 & 0.190 & 0.197 \\
WFM-1 db2 & 0.449 & 1.144 & 1.427 & 0.005 & 0.036 & \underline{0.072} & 0.173 & 0.239 & 0.261 \\
WFM-1 db4 & 0.532 & 1.079 & 1.228 & 0.007 & 0.041 & 0.072 & 0.326 & 0.387 & 0.388 \\
WFM-1 db6 & 0.451 & \underline{1.036} & 1.197 & 0.005 & 0.040 & 0.091 & 0.175 & 0.222 & 0.232 \\
WFM-3 haar & \textbf{0.415} & 1.073 & \underline{1.173} & \underline{0.004} & 0.036 & 0.077 & \underline{0.079} & \underline{0.169} & \textbf{0.191} \\
WFM-3 db2 & 0.450 & 1.143 & 1.227 & \textbf{0.003} & \textbf{0.032

## A.3  Spectral Coherence RMSE -- WFM + Generative Models

Values averaged over all rollout horizons within each PDE. Columns: Low / Mid / High bands.

In [150]:
t_spectral = build_spectral_table(models=WFM_all + gen_models)
t_spectral.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

In [151]:
# Rebuild with (pde_key, band) MultiIndex columns so to_latex_wide can look up PDE_LABELS
_label_to_key = {v: k for k, v in PDE_LABELS.items()}
_spec_wide = build_spectral_table(models=WFM_all + gen_models)
#_idx_order  = [labels.get(m, m) for m in WFM_all + gen_models]

# _spec_wide is (PDE_label, model_label) × [Low, Mid, High].
# Unstack the PDE level into columns → model × (band, PDE_label),
# then swap so PDE is the outer level → (PDE_label, band).
_spec_wide = _spec_wide.unstack(level=0).swaplevel(axis=1)

#_spec_wide.index = [labels.get(m, m) for m in WFM_all + gen_models if labels.get(m, m) in _spec_wide.index]
#_spec_wide.columns = pd.MultiIndex.from_tuples(
#    [(_label_to_key.get(p, p), b) for p, b in _spec_wide.columns],
#    names=["pde", "horizon"],
#)
# Remap PDE display labels → pde keys so to_latex_wide can call PDE_LABELS[pde]
#_spec_wide.columns = pd.MultiIndex.from_tuples(
#    [(_label_to_key.get(p, p), b) for p, b in _spec_wide.columns],
#    names=["pde", "horizon"],
#)

# Restore canonical model-display-label ordering
_model_label_order = [labels[m] for m in WFM_all + gen_models if labels[m] in _spec_wide.index]
#_spec_wide = _spec_wide.reindex(_model_label_order)
_spec_wide = _spec_wide.reindex(
    columns=pd.MultiIndex.from_tuples(
        [(PDE_LABELS[pde], band) for pde in PDEs for band in BANDS],
    )
)
_spec_wide.columns = pd.MultiIndex.from_tuples(
    [(_label_to_key.get(p, p), b) for p, b in _spec_wide.columns],
    names=["pde", "horizon"],
)
_spec_wide.index.name = "Model"

print(to_latex_wide(
    _spec_wide,
    caption=r"Spectral coherence RMSE $\downarrow$ (horizon-averaged) for Low / Mid / High frequency bands.",
    label="tab:appendix_spectral",
    model_groups=_crps_groups,
    row_label="Band",
))

\begin{table}[ht]
\centering
\caption{Spectral coherence RMSE $\downarrow$ (horizon-averaged) for Low / Mid / High frequency bands.}
\label{tab:appendix_spectral}
\begin{tabular}{l ccc ccc ccc}
\toprule
Band & \multicolumn{3}{c}{Turbulence} & \multicolumn{3}{c}{Rayleigh--B\'{e}nard} & \multicolumn{3}{c}{Active Matter} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
 & Low & Mid & High & Low & Mid & High & Low & Mid & High \\
\midrule
WFM-1 haar & 0.134 & 0.176 & \textbf{0.134} & 0.169 & 0.132 & 0.117 & 0.228 & 0.291 & 0.204 \\
WFM-1 db2 & 0.137 & 0.180 & 0.141 & 0.167 & 0.126 & 0.105 & 0.291 & 0.336 & 0.236 \\
WFM-1 db4 & \textbf{0.129} & 0.175 & 0.139 & 0.166 & 0.117 & \underline{0.077} & 0.307 & 0.351 & 0.251 \\
WFM-1 db6 & 0.134 & 0.176 & 0.141 & 0.188 & 0.125 & 0.085 & 0.283 & 0.347 & 0.242 \\
WFM-3 haar & \underline{0.132} & \textbf{0.173} & 0.141 & 0.150 & 0.130 & 0.114 & 0.205 & 0.279 & 0.256 \\
WFM-3 db2 & 0.140 & 0.187 & \underline{0.136} & \textbf{0.143} & 0.114 

## A.3b  Spectral Coherence RMSE -- by Band x Horizon (detailed)

Optional detailed breakdown: columns are a (band, horizon) MultiIndex.

In [152]:
t_spec_hz = build_spectral_horizon_table(models=WFM_all + gen_models)
t_spec_hz.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

## A.4  Wasserstein-1 Distance -- WFM + Generative Models

In [153]:
t_w1 = build_w1_table(models=WFM_all + gen_models)
t_w1.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

In [154]:
# Reshape: model rows x (pde_key, "$W_1$") MultiIndex columns
_w1_wide = t_w1[r"$W_1$"].unstack(level=0)
_w1_wide.columns = pd.MultiIndex.from_tuples(
    [(_label_to_key.get(c, c), r"$W_1$") for c in _w1_wide.columns],
    names=["pde", "horizon"],
)
_w1_wide.index.name = "Model"
print(to_latex_wide(
    _w1_wide,
    caption=r"Wasserstein-1 distance $W_1$ $\downarrow$ for generative and WFM models.",
    label="tab:appendix_w1",
    model_groups=_crps_groups,
    row_label="",
))

\begin{table}[ht]
\centering
\caption{Wasserstein-1 distance $W_1$ $\downarrow$ for generative and WFM models.}
\label{tab:appendix_w1}
\begin{tabular}{l c c c}
\toprule
 & \multicolumn{1}{c}{Active Matter} & \multicolumn{1}{c}{Rayleigh--B\'{e}nard} & \multicolumn{1}{c}{Turbulence} \\
\cmidrule(lr){2-2} \cmidrule(lr){3-3} \cmidrule(lr){4-4}
 & $W_1$ & $W_1$ & $W_1$ \\
\midrule
WFM-1 haar & 0.071 & 0.018 & 0.508 \\
WFM-1 db2 & 0.234 & 0.019 & 0.665 \\
WFM-1 db4 & 0.493 & 0.025 & 0.567 \\
WFM-1 db6 & 0.210 & 0.037 & \underline{0.493} \\
WFM-3 haar & \underline{0.036} & 0.022 & 0.504 \\
WFM-3 db2 & 0.045 & 0.019 & 0.631 \\
WFM-3 db4 & 0.146 & \textbf{0.017} & \textbf{0.403} \\
WFM-3 db6 & 0.140 & 0.026 & 0.532 \\
\midrule
FM$_\text{fourier}$ & 0.105 & 0.046 & 1.206 \\
WDNO & 0.178 & 0.110 & 3.530 \\
\midrule
FM$_\text{pixel}$ & \textbf{0.033} & \underline{0.018} & 0.504 \\
\bottomrule
\end{tabular}
\end{table}


---
# B. Ablation Tables

**Rows**: model display labels (WFM-1 variants, WFM-3 variants, generative baselines).
**Columns**: values of the physical parameter being swept.
Three metrics per parameter: **VRMSE** (all models), **CRPS** (generative + WFM), **Spectral RMSE** by band.

| Section | PDE              | Parameter   | New vs original |
|---------|------------------|-------------|-----------------|
| B.1     | Turbulence       | tcool       | + VRMSE (new)   |
| B.2     | Rayleigh–Bénard  | Ra          | + VRMSE (new)   |
| B.3     | Rayleigh–Bénard  | Pr          | + VRMSE (new)   |
| B.4     | Active Matter    | alpha       | + VRMSE (new)   |
| B.5     | Active Matter    | zeta        | + VRMSE (new)   |

## B.1  Turbulent Radiative Layer — $t_\mathrm{cool}$

In [155]:
_PDE  = "turbulent_radiative_layer_2D"
_PAR  = "tcool"
#_FILT = lambda v: v <= 0.030001   # keep only the three standard tcool values
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR) #, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"VRMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_vrmse_tcool",
))

tcool,0.030000,0.060000,0.100000,0.180000,0.320000,0.560000,1.000000,1.780000,3.160000
Model,,,,,,,,,
WFM-1 haar,0.669,0.673,0.657,0.641,0.647,0.710,0.656,0.963,0.669
WFM-1 db2,0.672,0.676,0.652,0.677,0.670,0.670,0.700,0.705,1.075
WFM-1 db4,0.664,0.661,0.632,0.696,0.802,0.714,0.678,0.719,0.631
WFM-1 db6,0.670,0.658,0.658,0.656,0.709,0.689,0.628,0.708,0.745
WFM-3 haar,0.659,0.684,0.707,0.690,0.656,0.795,0.814,1.051,0.627
WFM-3 db2,0.892,0.723,0.726,0.843,0.766,0.721,0.712,1.287,0.724
WFM-3 db4,0.678,0.676,0.640,0.653,0.642,0.823,0.667,0.908,0.795
WFM-3 db6,0.655,0.670,0.660,0.652,0.659,0.757,0.887,0.982,0.973
FM$_\text{pixel}$,0.668,0.676,0.701,0.694,0.708,0.792,0.749,1.314,0.790


\begin{table}[ht]
\centering
\caption{VRMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.}
\label{tab:ablation_trl_vrmse_tcool}
\begin{tabular}{lccccccccc}
\toprule
tcool & 0.0299999993294477 & 0.0599999986588954 & 0.1000000014901161 & 0.1800000071525573 & 0.3199999928474426 & 0.5600000023841858 & 1.0 & 1.7799999713897705 & 3.1600000858306885 \\
\midrule
WFM-1 haar & 0.669 & 0.673 & 0.657 & \textbf{0.641} & \underline{0.647} & 0.710 & 0.656 & 0.963 & 0.669 \\
WFM-1 db2 & 0.672 & 0.676 & 0.652 & 0.677 & 0.670 & \textbf{0.670} & 0.700 & \textbf{0.705} & 1.075 \\
WFM-1 db4 & 0.664 & \underline{0.661} & \textbf{0.632} & 0.696 & 0.802 & 0.714 & 0.678 & 0.719 & \underline{0.631} \\
WFM-1 db6 & 0.670 & \textbf{0.658} & 0.658 & 0.656 & 0.709 & \underline{0.689} & \underline{0.628} & \underline{0.708} & 0.745 \\
WFM-3 haar & 0.659 & 0.684 & 0.707 & 0.690 & 0.656 & 0.795 & 0.814 & 1.051 & \textbf{0.627} \\
WFM-3 db2 & 0.892 & 0.723 & 0.726 & 0.843 & 0.766 & 0.721 & 0.712 & 1.287 & 0.724 \\


In [156]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR) #, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"CRPS $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_crps_tcool",
))

tcool,0.030000,0.060000,0.100000,0.180000,0.320000,0.560000,1.000000,1.780000,3.160000
Model,,,,,,,,,
WFM-1 haar,1.122,1.124,0.816,1.085,0.831,0.790,0.812,0.994,0.688
WFM-1 db2,1.188,1.150,0.903,1.149,0.854,0.839,0.953,0.997,1.066
WFM-1 db4,1.126,1.127,0.861,1.159,0.839,0.874,0.841,1.012,0.702
WFM-1 db6,1.089,1.081,0.869,1.058,0.811,0.780,0.754,1.047,0.588
WFM-3 haar,1.085,1.149,0.932,1.047,0.732,0.775,0.782,0.944,0.562
WFM-3 db2,1.267,1.121,0.918,1.118,0.863,0.769,0.756,1.020,0.655
WFM-3 db4,1.031,1.038,0.808,0.992,0.782,0.742,0.709,0.897,0.657
WFM-3 db6,1.076,1.074,0.891,0.947,0.867,0.829,0.778,0.975,0.583
FM$_\text{pixel}$,1.161,1.145,0.909,1.064,0.926,0.799,0.759,0.968,0.551


\begin{table}[ht]
\centering
\caption{CRPS $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.}
\label{tab:ablation_trl_crps_tcool}
\begin{tabular}{lccccccccc}
\toprule
tcool & 0.0299999993294477 & 0.0599999986588954 & 0.1000000014901161 & 0.1800000071525573 & 0.3199999928474426 & 0.5600000023841858 & 1.0 & 1.7799999713897705 & 3.1600000858306885 \\
\midrule
WFM-1 haar & 1.122 & 1.124 & \underline{0.816} & 1.085 & 0.831 & 0.790 & 0.812 & 0.994 & 0.688 \\
WFM-1 db2 & 1.188 & 1.150 & 0.903 & 1.149 & 0.854 & 0.839 & 0.953 & 0.997 & 1.066 \\
WFM-1 db4 & 1.126 & 1.127 & 0.861 & 1.159 & 0.839 & 0.874 & 0.841 & 1.012 & 0.702 \\
WFM-1 db6 & 1.089 & 1.081 & 0.869 & 1.058 & 0.811 & 0.780 & \underline{0.754} & 1.047 & 0.588 \\
WFM-3 haar & 1.085 & 1.149 & 0.932 & 1.047 & \textbf{0.732} & 0.775 & 0.782 & \underline{0.944} & \underline{0.562} \\
WFM-3 db2 & 1.267 & 1.121 & 0.918 & 1.118 & 0.863 & \underline{0.769} & 0.756 & 1.020 & 0.655 \\
WFM-3 db4 & \textbf{1.031} & \textbf{1.038} & \textbf{0.808

In [157]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR) #, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
#print(to_latex_ablation(
print(to_latex_spectral_ablation(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_spectral_tcool",
))

\begin{table}[ht]
\centering
\caption{Spectral coherence RMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.}
\label{tab:ablation_trl_spectral_tcool}
\begin{tabular}{llccccccccc}
\toprule
Model & Band & 0.0299999993294477 & 0.0599999986588954 & 0.1000000014901161 & 0.1800000071525573 & 0.3199999928474426 & 0.5600000023841858 & 1.0 & 1.7799999713897705 & 3.1600000858306885 \\
\midrule
\multirow{3}{*}{WFM-1 haar} & Low & 0.137 & 0.143 & \underline{0.135} & 0.137 & \underline{0.135} & \textbf{0.124} & 0.132 & 0.145 & 0.123 \\
 & Mid & 0.191 & 0.197 & \underline{0.186} & 0.191 & 0.185 & \underline{0.170} & 0.159 & 0.154 & 0.149 \\
 & High & 0.142 & 0.142 & \textbf{0.132} & \textbf{0.139} & \underline{0.147} & \underline{0.145} & 0.134 & \underline{0.120} & 0.106 \\
\midrule
\multirow{3}{*}{WFM-1 db2} & Low & 0.140 & 0.140 & 0.137 & \underline{0.133} & 0.137 & 0.126 & 0.135 & 0.138 & 0.149 \\
 & Mid & 0.190 & 0.195 & 0.189 & \underline{0.189} & 0.183 & 0.176 & 0.168 & 0.164 & 0.163 \\
 

## B.2  Rayleigh–Bénard — Rayleigh Number $Ra$

Prandtl number aggregated out (mean over all Pr values in the sweep).

In [158]:
_PDE = "rayleigh_benard"
_PAR = "Rayleigh"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"VRMSE $\downarrow$ vs Rayleigh number $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_vrmse_ra",
))

Rayleigh,13.815511,18.420681,23.025850
Model,,,
WFM-1 haar,1.510,1.179,0.792
WFM-1 db2,1.414,1.003,0.769
WFM-1 db4,1.758,1.080,0.782
WFM-1 db6,1.512,1.129,1.033
WFM-3 haar,1.793,0.989,0.775
WFM-3 db2,1.307,0.828,0.761
WFM-3 db4,1.498,0.767,0.714
WFM-3 db6,1.314,0.761,0.770
FM$_\text{pixel}$,1.352,1.084,0.792


\begin{table}[ht]
\centering
\caption{VRMSE $\downarrow$ vs Rayleigh number $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).}
\label{tab:ablation_rb_vrmse_ra}
\begin{tabular}{lccc}
\toprule
Rayleigh & 13.815510749816896 & 18.42068099975586 & 23.025850296020508 \\
\midrule
WFM-1 haar & 1.510 & 1.179 & 0.792 \\
WFM-1 db2 & 1.414 & 1.003 & 0.769 \\
WFM-1 db4 & 1.758 & 1.080 & 0.782 \\
WFM-1 db6 & 1.512 & 1.129 & 1.033 \\
WFM-3 haar & 1.793 & 0.989 & 0.775 \\
WFM-3 db2 & \underline{1.307} & 0.828 & \underline{0.761} \\
WFM-3 db4 & 1.498 & \underline{0.767} & \textbf{0.714} \\
WFM-3 db6 & 1.314 & \textbf{0.761} & 0.770 \\
FM$_\text{pixel}$ & 1.352 & 1.084 & 0.792 \\
FM$_\text{fourier}$ & \textbf{0.795} & 1.036 & 0.798 \\
WDNO & 8.640 & 5.080 & 2.604 \\
\bottomrule
\end{tabular}
\end{table}


In [159]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"CRPS $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_crps_ra",
))

Rayleigh,13.815511,18.420681,23.025850
Model,,,
WFM-1 haar,0.058,0.065,0.054
WFM-1 db2,0.058,0.057,0.052
WFM-1 db4,0.058,0.059,0.052
WFM-1 db6,0.062,0.068,0.076
WFM-3 haar,0.059,0.063,0.054
WFM-3 db2,0.058,0.057,0.053
WFM-3 db4,0.054,0.057,0.048
WFM-3 db6,0.059,0.056,0.055
FM$_\text{pixel}$,0.056,0.066,0.055


\begin{table}[ht]
\centering
\caption{CRPS $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).}
\label{tab:ablation_rb_crps_ra}
\begin{tabular}{lccc}
\toprule
Rayleigh & 13.815510749816896 & 18.42068099975586 & 23.025850296020508 \\
\midrule
WFM-1 haar & 0.058 & 0.065 & 0.054 \\
WFM-1 db2 & 0.058 & \underline{0.057} & \underline{0.052} \\
WFM-1 db4 & 0.058 & 0.059 & 0.052 \\
WFM-1 db6 & 0.062 & 0.068 & 0.076 \\
WFM-3 haar & 0.059 & 0.063 & 0.054 \\
WFM-3 db2 & 0.058 & 0.057 & 0.053 \\
WFM-3 db4 & \textbf{0.054} & 0.057 & \textbf{0.048} \\
WFM-3 db6 & 0.059 & \textbf{0.056} & 0.055 \\
FM$_\text{pixel}$ & \underline{0.056} & 0.066 & 0.055 \\
FM$_\text{fourier}$ & 0.071 & 0.081 & 0.064 \\
WDNO & 0.120 & 0.118 & 0.116 \\
\bottomrule
\end{tabular}
\end{table}


In [99]:
#np.exp(13.815510749816896), np.exp(18.4206809997558), np.exp(23.025850296020508)
#np.exp(-2.3025851249694824), np.exp(0), np.exp(2.302585092994046)

In [160]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
#print(to_latex_ablation(
print(to_latex_spectral_ablation(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_spectral_ra",
))

\begin{table}[ht]
\centering
\caption{Spectral coherence RMSE $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).}
\label{tab:ablation_rb_spectral_ra}
\begin{tabular}{llccc}
\toprule
Model & Band & 13.815510749816896 & 18.42068099975586 & 23.025850296020508 \\
\midrule
\multirow{3}{*}{WFM-1 haar} & Low & 0.172 & 0.158 & 0.135 \\
 & Mid & 0.189 & 0.136 & 0.132 \\
 & High & 0.112 & 0.188 & 0.163 \\
\midrule
\multirow{3}{*}{WFM-1 db2} & Low & 0.163 & \underline{0.157} & 0.143 \\
 & Mid & 0.165 & 0.131 & 0.133 \\
 & High & 0.094 & 0.163 & 0.151 \\
\midrule
\multirow{3}{*}{WFM-1 db4} & Low & 0.150 & 0.169 & 0.141 \\
 & Mid & \textbf{0.116} & 0.130 & 0.132 \\
 & High & \textbf{0.048} & 0.102 & \textbf{0.120} \\
\midrule
\multirow{3}{*}{WFM-1 db6} & Low & 0.184 & 0.190 & 0.209 \\
 & Mid & 0.141 & 0.125 & 0.177 \\
 & High & 0.053 & \textbf{0.090} & 0.181 \\
\midrule
\multirow{3}{*}{WFM-3 haar} & Low & 0.157 & 0.167 & \underline{0.129} \\
 & Mid & 0.186 & 0.133 & 0.129 \\
 & High &

## B.3  Rayleigh–Bénard — Prandtl Number $Pr$

Rayleigh number aggregated out (mean over all Ra values in the sweep).

In [161]:
_PDE = "rayleigh_benard"
_PAR = "Prandtl"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"VRMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_vrmse_pr",
))

Prandtl,-2.302585,0.000000,2.302585
Model,,,
WFM-1 haar,1.491,0.921,1.070
WFM-1 db2,1.195,0.886,1.106
WFM-1 db4,1.344,1.070,1.207
WFM-1 db6,1.315,1.104,1.256
WFM-3 haar,1.410,0.921,1.226
WFM-3 db2,1.115,0.866,0.915
WFM-3 db4,1.144,0.873,0.961
WFM-3 db6,0.917,0.836,1.091
FM$_\text{pixel}$,1.188,0.953,1.088


\begin{table}[ht]
\centering
\caption{VRMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).}
\label{tab:ablation_rb_vrmse_pr}
\begin{tabular}{lccc}
\toprule
Prandtl & -2.3025851249694824 & 0.0 & 2.3025851249694824 \\
\midrule
WFM-1 haar & 1.491 & 0.921 & 1.070 \\
WFM-1 db2 & 1.195 & 0.886 & 1.106 \\
WFM-1 db4 & 1.344 & 1.070 & 1.207 \\
WFM-1 db6 & 1.315 & 1.104 & 1.256 \\
WFM-3 haar & 1.410 & 0.921 & 1.226 \\
WFM-3 db2 & 1.115 & 0.866 & \underline{0.915} \\
WFM-3 db4 & 1.144 & 0.873 & 0.961 \\
WFM-3 db6 & \textbf{0.917} & \underline{0.836} & 1.091 \\
FM$_\text{pixel}$ & 1.188 & 0.953 & 1.088 \\
FM$_\text{fourier}$ & \underline{1.072} & \textbf{0.762} & \textbf{0.796} \\
WDNO & 4.491 & 3.943 & 7.889 \\
\bottomrule
\end{tabular}
\end{table}


In [162]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"CRPS $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_crps_pr",
))

Prandtl,-2.302585,0.000000,2.302585
Model,,,
WFM-1 haar,0.094,0.054,0.029
WFM-1 db2,0.083,0.052,0.031
WFM-1 db4,0.085,0.050,0.034
WFM-1 db6,0.093,0.065,0.049
WFM-3 haar,0.095,0.049,0.032
WFM-3 db2,0.086,0.051,0.031
WFM-3 db4,0.082,0.049,0.028
WFM-3 db6,0.083,0.048,0.039
FM$_\text{pixel}$,0.090,0.054,0.033


\begin{table}[ht]
\centering
\caption{CRPS $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).}
\label{tab:ablation_rb_crps_pr}
\begin{tabular}{lccc}
\toprule
Prandtl & -2.3025851249694824 & 0.0 & 2.3025851249694824 \\
\midrule
WFM-1 haar & 0.094 & 0.054 & \underline{0.029} \\
WFM-1 db2 & \underline{0.083} & 0.052 & 0.031 \\
WFM-1 db4 & 0.085 & 0.050 & 0.034 \\
WFM-1 db6 & 0.093 & 0.065 & 0.049 \\
WFM-3 haar & 0.095 & \underline{0.049} & 0.032 \\
WFM-3 db2 & 0.086 & 0.051 & 0.031 \\
WFM-3 db4 & \textbf{0.082} & 0.049 & \textbf{0.028} \\
WFM-3 db6 & 0.083 & \textbf{0.048} & 0.039 \\
FM$_\text{pixel}$ & 0.090 & 0.054 & 0.033 \\
FM$_\text{fourier}$ & 0.114 & 0.058 & 0.044 \\
WDNO & 0.145 & 0.112 & 0.097 \\
\bottomrule
\end{tabular}
\end{table}


In [163]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
#print(to_latex_ablation(
print(to_latex_spectral_ablation(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_spectral_pr",
))

\begin{table}[ht]
\centering
\caption{Spectral coherence RMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).}
\label{tab:ablation_rb_spectral_pr}
\begin{tabular}{llccc}
\toprule
Model & Band & -2.3025851249694824 & 0.0 & 2.3025851249694824 \\
\midrule
\multirow{3}{*}{WFM-1 haar} & Low & 0.175 & 0.142 & 0.147 \\
 & Mid & 0.159 & 0.165 & 0.133 \\
 & High & 0.193 & 0.160 & 0.110 \\
\midrule
\multirow{3}{*}{WFM-1 db2} & Low & 0.169 & 0.147 & 0.146 \\
 & Mid & 0.151 & 0.155 & 0.124 \\
 & High & 0.182 & 0.138 & 0.089 \\
\midrule
\multirow{3}{*}{WFM-1 db4} & Low & 0.178 & 0.138 & 0.144 \\
 & Mid & 0.138 & \textbf{0.130} & 0.111 \\
 & High & 0.121 & \textbf{0.086} & \textbf{0.063} \\
\midrule
\multirow{3}{*}{WFM-1 db6} & Low & 0.212 & 0.183 & 0.188 \\
 & Mid & 0.151 & 0.151 & 0.142 \\
 & High & 0.130 & 0.118 & 0.075 \\
\midrule
\multirow{3}{*}{WFM-3 haar} & Low & 0.182 & 0.136 & 0.134 \\
 & Mid & 0.156 & 0.157 & 0.135 \\
 & High & 0.189 & 0.150 & 0.108 \\
\midrule
\multirow{3

## B.4  Active Matter — $\alpha$

Zeta aggregated out. Filtered to the three main sweep values: $\alpha \in \{-3, -4, -5\}$.

In [164]:
_PDE  = "active_matter"
_PAR  = "alpha"
_FILT = lambda v: v in [-1, -2, -3, -4, -5]
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"VRMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_vrmse_alpha",
))

alpha,-5.000000,-4.000000,-3.000000,-2.000000,-1.000000
Model,,,,,
WFM-1 haar,0.957,0.960,0.974,1.100,3.945
WFM-1 db2,1.262,1.267,1.308,1.722,8.582
WFM-1 db4,1.844,1.909,1.997,2.945,20.368
WFM-1 db6,1.201,1.219,1.272,1.629,9.967
WFM-3 haar,0.914,0.913,0.902,0.831,1.793
WFM-3 db2,0.936,0.945,0.938,0.881,1.932
WFM-3 db4,1.133,1.212,1.165,1.210,5.460
WFM-3 db6,1.070,1.057,1.039,1.091,4.346
FM$_\text{pixel}$,0.877,0.874,0.861,0.775,1.824


\begin{table}[ht]
\centering
\caption{VRMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).}
\label{tab:ablation_am_vrmse_alpha}
\begin{tabular}{lccccc}
\toprule
alpha & -5.0 & -4.0 & -3.0 & -2.0 & -1.0 \\
\midrule
WFM-1 haar & 0.957 & 0.960 & 0.974 & 1.100 & 3.945 \\
WFM-1 db2 & 1.262 & 1.267 & 1.308 & 1.722 & 8.582 \\
WFM-1 db4 & 1.844 & 1.909 & 1.997 & 2.945 & 20.368 \\
WFM-1 db6 & 1.201 & 1.219 & 1.272 & 1.629 & 9.967 \\
WFM-3 haar & \underline{0.914} & \underline{0.913} & \underline{0.902} & \underline{0.831} & \underline{1.793} \\
WFM-3 db2 & 0.936 & 0.945 & 0.938 & 0.881 & 1.932 \\
WFM-3 db4 & 1.133 & 1.212 & 1.165 & 1.210 & 5.460 \\
WFM-3 db6 & 1.070 & 1.057 & 1.039 & 1.091 & 4.346 \\
FM$_\text{pixel}$ & \textbf{0.877} & \textbf{0.874} & \textbf{0.861} & \textbf{0.775} & 1.824 \\
FM$_\text{fourier}$ & 0.958 & 0.998 & 1.034 & 1.020 & \textbf{1.499} \\
WDNO & 1.188 & 1.209 & 1.221 & 1.490 & 6.246 \\
\bottomrule
\end{tabular}
\end{table}


In [165]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"CRPS $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_crps_alpha",
))

alpha,-5.000000,-4.000000,-3.000000,-2.000000,-1.000000
Model,,,,,
WFM-1 haar,0.290,0.225,0.191,0.147,0.106
WFM-1 db2,0.339,0.273,0.244,0.209,0.187
WFM-1 db4,0.443,0.393,0.377,0.359,0.344
WFM-1 db6,0.304,0.248,0.225,0.193,0.174
WFM-3 haar,0.281,0.217,0.175,0.126,0.087
WFM-3 db2,0.284,0.221,0.180,0.129,0.098
WFM-3 db4,0.316,0.265,0.214,0.162,0.137
WFM-3 db6,0.296,0.232,0.193,0.154,0.127
FM$_\text{pixel}$,0.272,0.209,0.169,0.121,0.099


\begin{table}[ht]
\centering
\caption{CRPS $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).}
\label{tab:ablation_am_crps_alpha}
\begin{tabular}{lccccc}
\toprule
alpha & -5.0 & -4.0 & -3.0 & -2.0 & -1.0 \\
\midrule
WFM-1 haar & 0.290 & 0.225 & 0.191 & 0.147 & 0.106 \\
WFM-1 db2 & 0.339 & 0.273 & 0.244 & 0.209 & 0.187 \\
WFM-1 db4 & 0.443 & 0.393 & 0.377 & 0.359 & 0.344 \\
WFM-1 db6 & 0.304 & 0.248 & 0.225 & 0.193 & 0.174 \\
WFM-3 haar & \underline{0.281} & \underline{0.217} & \underline{0.175} & \underline{0.126} & \textbf{0.087} \\
WFM-3 db2 & 0.284 & 0.221 & 0.180 & 0.129 & \underline{0.098} \\
WFM-3 db4 & 0.316 & 0.265 & 0.214 & 0.162 & 0.137 \\
WFM-3 db6 & 0.296 & 0.232 & 0.193 & 0.154 & 0.127 \\
FM$_\text{pixel}$ & \textbf{0.272} & \textbf{0.209} & \textbf{0.169} & \textbf{0.121} & 0.099 \\
FM$_\text{fourier}$ & 0.318 & 0.260 & 0.231 & 0.184 & 0.121 \\
WDNO & 0.382 & 0.312 & 0.267 & 0.209 & 0.167 \\
\bottomrule
\end{tabular}
\end{table}


In [166]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_spectral_alpha",
))

\begin{table}[ht]
\centering
\caption{Spectral coherence RMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).}
\label{tab:ablation_am_spectral_alpha}
\begin{tabular}{lccccccccccccccc}
\toprule
 & ('Low', -5.0) & ('Low', -4.0) & ('Low', -3.0) & ('Low', -2.0) & ('Low', -1.0) & ('Mid', -5.0) & ('Mid', -4.0) & ('Mid', -3.0) & ('Mid', -2.0) & ('Mid', -1.0) & ('High', -5.0) & ('High', -4.0) & ('High', -3.0) & ('High', -2.0) & ('High', -1.0) \\
\midrule
WFM-1 haar & 0.187 & 0.186 & 0.201 & 0.211 & 0.375 & \textbf{0.236} & 0.224 & 0.253 & 0.276 & 0.450 & \underline{0.202} & \underline{0.244} & \underline{0.232} & \underline{0.171} & 0.182 \\
WFM-1 db2 & 0.254 & 0.245 & 0.266 & 0.282 & 0.424 & 0.285 & 0.279 & 0.310 & 0.323 & 0.488 & 0.222 & 0.312 & 0.281 & 0.197 & 0.210 \\
WFM-1 db4 & 0.256 & 0.253 & 0.273 & 0.281 & 0.430 & 0.299 & 0.298 & 0.321 & 0.323 & 0.491 & 0.226 & 0.325 & 0.292 & 0.205 & 0.221 \\
WFM-1 db6 & 0.226 & 0.225 & 0.253 & 0.261 & 0.430 & 0.297 & 0.296 & 0.323

## B.5  Active Matter — $\zeta$

Alpha aggregated out (mean over all alpha values in the sweep).

In [167]:
_PDE = "active_matter"
_PAR = "zeta"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"VRMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_vrmse_zeta",
))

zeta,1.000000,5.000000,11.000000,17.000000
Model,,,,
WFM-1 haar,4.098,0.961,0.993,0.969
WFM-1 db2,8.845,1.351,1.427,1.300
WFM-1 db4,21.020,2.060,2.263,1.974
WFM-1 db6,10.165,1.286,1.382,1.279
WFM-3 haar,1.811,0.877,0.860,0.916
WFM-3 db2,1.885,0.907,0.938,0.951
WFM-3 db4,5.455,1.140,1.213,1.209
WFM-3 db6,4.347,1.048,1.074,1.066
FM$_\text{pixel}$,1.828,0.830,0.823,0.872


\begin{table}[ht]
\centering
\caption{VRMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).}
\label{tab:ablation_am_vrmse_zeta}
\begin{tabular}{lcccc}
\toprule
zeta & 1.0 & 5.0 & 11.0 & 17.0 \\
\midrule
WFM-1 haar & 4.098 & 0.961 & 0.993 & 0.969 \\
WFM-1 db2 & 8.845 & 1.351 & 1.427 & 1.300 \\
WFM-1 db4 & 21.020 & 2.060 & 2.263 & 1.974 \\
WFM-1 db6 & 10.165 & 1.286 & 1.382 & 1.279 \\
WFM-3 haar & \underline{1.811} & \underline{0.877} & \underline{0.860} & \underline{0.916} \\
WFM-3 db2 & 1.885 & 0.907 & 0.938 & 0.951 \\
WFM-3 db4 & 5.455 & 1.140 & 1.213 & 1.209 \\
WFM-3 db6 & 4.347 & 1.048 & 1.074 & 1.066 \\
FM$_\text{pixel}$ & 1.828 & \textbf{0.830} & \textbf{0.823} & \textbf{0.872} \\
FM$_\text{fourier}$ & \textbf{1.474} & 0.987 & 1.042 & 1.018 \\
WDNO & 6.435 & 1.221 & 1.302 & 1.246 \\
\bottomrule
\end{tabular}
\end{table}


In [168]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"CRPS $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_crps_zeta",
))

zeta,1.000000,5.000000,11.000000,17.000000
Model,,,,
WFM-1 haar,0.113,0.216,0.173,0.200
WFM-1 db2,0.187,0.271,0.232,0.252
WFM-1 db4,0.350,0.398,0.369,0.376
WFM-1 db6,0.174,0.246,0.213,0.231
WFM-3 haar,0.097,0.200,0.153,0.193
WFM-3 db2,0.096,0.202,0.164,0.197
WFM-3 db4,0.137,0.238,0.199,0.232
WFM-3 db6,0.134,0.219,0.178,0.208
FM$_\text{pixel}$,0.104,0.191,0.151,0.186


\begin{table}[ht]
\centering
\caption{CRPS $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).}
\label{tab:ablation_am_crps_zeta}
\begin{tabular}{lcccc}
\toprule
zeta & 1.0 & 5.0 & 11.0 & 17.0 \\
\midrule
WFM-1 haar & 0.113 & 0.216 & 0.173 & 0.200 \\
WFM-1 db2 & 0.187 & 0.271 & 0.232 & 0.252 \\
WFM-1 db4 & 0.350 & 0.398 & 0.369 & 0.376 \\
WFM-1 db6 & 0.174 & 0.246 & 0.213 & 0.231 \\
WFM-3 haar & \underline{0.097} & \underline{0.200} & \underline{0.153} & \underline{0.193} \\
WFM-3 db2 & \textbf{0.096} & 0.202 & 0.164 & 0.197 \\
WFM-3 db4 & 0.137 & 0.238 & 0.199 & 0.232 \\
WFM-3 db6 & 0.134 & 0.219 & 0.178 & 0.208 \\
FM$_\text{pixel}$ & 0.104 & \textbf{0.191} & \textbf{0.151} & \textbf{0.186} \\
FM$_\text{fourier}$ & 0.132 & 0.247 & 0.211 & 0.238 \\
WDNO & 0.164 & 0.292 & 0.249 & 0.283 \\
\bottomrule
\end{tabular}
\end{table}


In [169]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_ablation(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_spectral_zeta",
))

\begin{table}[ht]
\centering
\caption{Spectral coherence RMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).}
\label{tab:ablation_am_spectral_zeta}
\begin{tabular}{lcccccccccccc}
\toprule
 & ('Low', 1.0) & ('Low', 5.0) & ('Low', 11.0) & ('Low', 17.0) & ('Mid', 1.0) & ('Mid', 5.0) & ('Mid', 11.0) & ('Mid', 17.0) & ('High', 1.0) & ('High', 5.0) & ('High', 11.0) & ('High', 17.0) \\
\midrule
WFM-1 haar & 0.339 & 0.190 & 0.231 & 0.186 & 0.421 & 0.249 & 0.284 & 0.223 & 0.106 & \underline{0.196} & 0.238 & \underline{0.257} \\
WFM-1 db2 & 0.387 & 0.263 & 0.297 & 0.242 & 0.458 & 0.302 & 0.336 & 0.276 & 0.122 & 0.212 & 0.288 & 0.333 \\
WFM-1 db4 & 0.387 & 0.266 & 0.303 & 0.252 & 0.451 & 0.309 & 0.348 & 0.294 & 0.131 & 0.216 & 0.304 & 0.342 \\
WFM-1 db6 & 0.379 & 0.242 & 0.283 & 0.231 & 0.462 & 0.312 & 0.350 & 0.294 & 0.125 & 0.211 & 0.289 & 0.335 \\
WFM-3 haar & 0.341 & 0.183 & 0.196 & 0.185 & 0.443 & 0.263 & 0.265 & 0.223 & 0.112 & 0.281 & 0.300 & 0.320 \\
WFM-3 db2 & 0.328 